Contoh 2.3 ALGORITMA BAYESIAN OPTIMIZATION

In [147]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Memasukkan keseluruhan data olahan
data_olahan = {'Temperatur': [80, 83, 68, 64, 69, 71, 78, 82, 73, 77],
               'Kelembapan': [90, 78, 80, 65, 70, 80, 75, 92, 88, 70],
               'Jumlah_Pemain': [39, 43, 28, 43, 56, 13, 51, 41, 29, 36]}
df = pd.DataFrame(data_olahan)

# Penetapan variabel input dan variabel output
X, y = df.drop('Jumlah_Pemain', axis=1), df['Jumlah_Pemain']

# Splitting data olahan menjadi data latih dan data uji dengan perbandingan proporsi data 50%:50%
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.5, shuffle=False)


In [148]:
from bayes_opt import BayesianOptimization
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import root_mean_squared_error

# Menentukan hyperparameter target yang akan dioptimalkan beserta batasan nilainya
pbounds = {'min_samples_split': (2, 5),
           'min_samples_leaf': (1,5)}

# Mendefinisikan fungsi representasi target hyperparameter yang akan dioptimalkan oleh algoritma Bayesian Optimization
def dt_cv_score(min_samples_split, min_samples_leaf):
    try:        
        min_samples_split = int(min_samples_split)
        min_samples_leaf = int(min_samples_leaf)

        model = DecisionTreeRegressor(min_samples_split=min_samples_split,                                     
                                      min_samples_leaf=min_samples_leaf,
                                      random_state=42)
        
        scores = cross_val_score(model,
                                 X_train,
                                 y_train,
                                 cv=3,
                                 scoring='neg_root_mean_squared_error')

        return np.mean(scores)

    except Exception as e:
        print(f"Error dengan parameter: {e}")
        return -1e9 

# Menjalankan algoritma Bayesian Optimization untuk mengoptimalkan nilai hyperparameter target
optimizer = BayesianOptimization(f=dt_cv_score,  # Fungsi yang akan dioptimalkan
                                 pbounds=pbounds,  # Batasan parameter
                                 random_state=42,
                                 verbose=2)

optimizer.maximize(init_points=5,
                   n_iter=20) 

best_params = optimizer.max['params']

best_params_formatted = {'min_samples_split': int(best_params['min_samples_split']),
                         'min_samples_leaf': int(best_params['min_samples_leaf'])}

# Melihat nilai hyperparameter optimal hasil algoritma Bayesian Optimization
print(f'Hyperparameter terbaik: {best_params_formatted}')

|   iter    |  target   | min_sa... | min_sa... |
-------------------------------------------------
| 1         | -11.01906 | 3.1236203 | 4.8028572 |
| 2         | -11.01906 | 4.1959818 | 3.3946339 |
| 3         | -15.99394 | 2.4680559 | 1.6239780 |
| 4         | -11.01906 | 2.1742508 | 4.4647045 |
| 5         | -11.01906 | 3.8033450 | 3.8322903 |
| 6         | -11.01906 | 5.0       | 5.0       |
| 7         | -11.01906 | 5.0       | 1.0       |
| 8         | -11.01906 | 5.0       | 2.4106854 |
| 9         | -11.01906 | 5.0       | 3.8271436 |
| 10        | -11.01906 | 2.0       | 5.0       |
| 11        | -11.01906 | 4.1151875 | 5.0       |
| 12        | -11.01906 | 4.3969963 | 4.2821637 |
| 13        | -11.01906 | 5.0       | 1.6464638 |
| 14        | -11.01906 | 5.0       | 3.1439918 |
| 15        | -11.01906 | 2.5450578 | 5.0       |
| 16        | -11.01906 | 2.9153197 | 4.2028894 |
| 17        | -11.01906 | 4.5050044 | 3.6792406 |
| 18        | -11.01906 | 3.6707137 | 4.4391610 |


In [149]:
# Melatih model Random Forest dengan nilai hyperparameter target sesuai hasil kinerja algoritma Bayesian Optimization
best_dt_model = DecisionTreeRegressor(**best_params_formatted,
                                      random_state=42)
best_dt_model.fit(X_train, y_train)

DecisionTreeRegressor(min_samples_leaf=2, min_samples_split=4, random_state=42)

In [150]:
# Model Random Forest dengan Bayesian Optimization Hyperparameter Tuning melakukan prediksi terhadap variabel target 'Close_Diff'
y_pred = best_dt_model.predict(X_test)

# Menampilkan hasil prediksi
print(y_pred)

# Evaluasi model RF dengan metrik RMSE
rmse = root_mean_squared_error(y_test, y_pred)
print(f'RMSE: {rmse:.4f}')

[33.5        47.33333333 33.5        33.5        47.33333333]
RMSE: 11.3017
